# Import latest MLflow traces and analyze errors/latency

This notebook imports the latest `N_TRACES` from one MLflow experiment, without filtering by run ID. It then uses the existing `hg-ds-evals` trace parser to surface:

- trace-level MLflow state,
- span-level errors hidden inside otherwise `OK` traces,
- LangGraph path / node sequence,
- latency buckets and likely hidden LLM retry/backoff,
- the longest traces and sessions.

## Configuration

In [1]:
import os
# Strip env vars the Databricks VS Code extension injects — its loopback OAuth
# broker URL goes stale on extension/VS Code restarts and hangs the kernel.
# Falling through to the .databrickscfg profile is more robust.
for v in ("DATABRICKS_AUTH_TYPE", "DATABRICKS_METADATA_SERVICE_URL",
          "DATABRICKS_HOST", "DATABRICKS_CLUSTER_ID"):
    os.environ.pop(v, None)
os.environ["DATABRICKS_CONFIG_PROFILE"] = "adb-uat"

In [2]:
from pathlib import Path

EXPERIMENT_ID = "2373724612324597"
N_TRACES = 100
DBX_PROFILE = "adb-uat"

# Keep generated analysis outside the repo so input/checkpoint files stay untouched.
OUT_DIR = Path("/private/tmp/hg_latest_trace_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Adjust only if your hg-ds-evals checkout lives somewhere else.
HG_DS_EVALS_ROOT = Path("/Users/SG7CB/Developer/hg-ds-evals")

print(f"Experiment: {EXPERIMENT_ID}")
print(f"Trace limit: {N_TRACES}")
print(f"Output dir:  {OUT_DIR}")

Experiment: 2373724612324597
Trace limit: 100
Output dir:  /private/tmp/hg_latest_trace_analysis


## Auth and imports

In [3]:
import os
import sys

# The Databricks VS Code extension can inject loopback OAuth variables that go stale.
# Clearing them lets the Databricks SDK use the normal CLI profile instead.
for name in (
    "DATABRICKS_AUTH_TYPE",
    "DATABRICKS_METADATA_SERVICE_URL",
    "DATABRICKS_HOST",
    "DATABRICKS_CLUSTER_ID",
):
    os.environ.pop(name, None)

os.environ["DATABRICKS_CONFIG_PROFILE"] = DBX_PROFILE

if not (HG_DS_EVALS_ROOT / "hg_ds_evals").is_dir():
    raise FileNotFoundError(f"Could not find hg_ds_evals package at {HG_DS_EVALS_ROOT}")

if str(HG_DS_EVALS_ROOT) not in sys.path:
    sys.path.insert(0, str(HG_DS_EVALS_ROOT))

import json
from collections import Counter
from typing import Any

import mlflow
import pandas as pd
from databricks.sdk import WorkspaceClient

from hg_ds_evals.preprocessing.latency import build_latency_dataframe
from hg_ds_evals.preprocessing.traces import build_dataframe_from_mlflow_traces

workspace = WorkspaceClient()
me = workspace.current_user.me()
mlflow.set_tracking_uri("databricks")

print(f"Authenticated as: {me.user_name}")
print(f"Workspace host:   {workspace.config.host}")
print(f"MLflow URI:       {mlflow.get_tracking_uri()}")

Authenticated as: sg7cb@s-mxs.net
Workspace host:   https://adb-7405614616397813.13.azuredatabricks.net
MLflow URI:       databricks


## Fetch the last N traces from the experiment

Important difference from the existing run-level notebooks: this call does **not** pass `run_id`. It asks MLflow for the newest traces in the experiment by `timestamp_ms DESC`.

In [4]:
traces_df = mlflow.search_traces(
    locations=[EXPERIMENT_ID],
    max_results=N_TRACES,
    order_by=["timestamp_ms DESC"],
)

raw_pickle = OUT_DIR / f"raw_latest_{N_TRACES}_traces.pkl"
traces_df.to_pickle(raw_pickle)

print(f"Fetched rows: {len(traces_df):,}")
print(f"Columns: {traces_df.columns.tolist()}")
print(f"Cached raw traces to: {raw_pickle}")
traces_df.head(3)

Fetched rows: 100
Columns: ['trace_id', 'trace', 'client_request_id', 'state', 'request_time', 'execution_duration', 'request', 'response', 'trace_metadata', 'tags', 'spans', 'assessments']
Cached raw traces to: /private/tmp/hg_latest_trace_analysis/raw_latest_100_traces.pkl


,trace_id,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-18048f6590c832eb461015cc55f3f8ab,"{""info"": {""trace_id"": ""tr-18048f6590c832eb4610...",tr-18048f6590c832eb461015cc55f3f8ab,OK,1780931868766,19323,{'messages': [{'content': 'How can I improve m...,{'messages': [{'content': 'How can I improve m...,"{'mlflow.user': 'cnb', 'mlflow.traceInputs': '...",{'mlflow.artifactLocation': 'dbfs:/databricks/...,"[{'trace_id': 'GASPZZDIMutGEBXMVfP4qw==', 'spa...",[]
1,tr-37c244d89e721032c224b6f79625225e,"{""info"": {""trace_id"": ""tr-37c244d89e721032c224...",tr-37c244d89e721032c224b6f79625225e,OK,1780931715586,19629,{'messages': [{'content': 'How can I improve m...,{'messages': [{'content': 'How can I improve m...,"{'mlflow.user': 'cnb', 'mlflow.traceInputs': '...",{'mlflow.artifactLocation': 'dbfs:/databricks/...,"[{'trace_id': 'N8JE2J5yEDLCJLb3liUiXg==', 'spa...",[]
2,tr-aed6070c65b3cbe6a23c7ea8ee7deadb,"{""info"": {""trace_id"": ""tr-aed6070c65b3cbe6a23c...",tr-aed6070c65b3cbe6a23c7ea8ee7deadb,OK,1780931396067,15959,{'messages': [{'content': 'What do I spend on ...,{'messages': [{'content': 'What do I spend on ...,"{'mlflow.user': 'cnb', 'mlflow.traceInputs': '...",{'mlflow.artifactLocation': 'dbfs:/databricks/...,"[{'trace_id': 'rtYHDGWzy+aiPH6o7n3q2w==', 'spa...",[]


## Parse traces, span errors, and latency

In [5]:
parse_result = build_dataframe_from_mlflow_traces(traces_df)
parsed_df = parse_result.dataframe.copy()
latency_df = build_latency_dataframe(traces_df).reset_index()

trace_level_df = parsed_df.merge(latency_df, on="trace_id", how="left", validate="one_to_one")

print(f"Parsed rows:  {len(parsed_df):,}")
print(f"Parse errors: {len(parse_result.parse_errors):,}")
print(f"Rows using trace_id as fallback test_case_id: {len(parse_result.untagged_trace_ids):,}")

if parse_result.parse_errors:
    for err in parse_result.parse_errors[:5]:
        print(f"  {err.trace_id}: {err.error}")

Parsed rows:  100
Parse errors: 0
Rows using trace_id as fallback test_case_id: 100


## Helper functions for readable diagnostics

In [6]:
def json_loads(value: Any, default: Any) -> Any:
    if value is None:
        return default
    if isinstance(value, (dict, list)):
        return value
    if isinstance(value, float) and pd.isna(value):
        return default
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return default
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return default
    return default


def ms_to_s(value: Any) -> float | None:
    try:
        if value is None or pd.isna(value):
            return None
        return round(float(value) / 1000.0, 3)
    except (TypeError, ValueError):
        return None


def span_type(span: dict[str, Any]) -> str:
    raw = (span.get("attributes") or {}).get("mlflow.spanType")
    if isinstance(raw, str):
        try:
            return str(json.loads(raw))
        except json.JSONDecodeError:
            return raw
    return str(raw or "")


def span_duration_ms(span: dict[str, Any]) -> float:
    try:
        start = int(span.get("start_time_unix_nano") or 0)
        end = int(span.get("end_time_unix_nano") or 0)
    except (TypeError, ValueError):
        return 0.0
    return max(0.0, (end - start) / 1e6)


def coerce_spans(value: Any) -> list[dict[str, Any]]:
    if isinstance(value, list):
        return [item for item in value if isinstance(item, dict)]
    parsed = json_loads(value, [])
    return [item for item in parsed if isinstance(item, dict)] if isinstance(parsed, list) else []


def short_text(value: Any, limit: int = 220) -> str:
    if value is None:
        return ""
    if not isinstance(value, str):
        try:
            value = json.dumps(value, ensure_ascii=False)
        except TypeError:
            value = str(value)
    text = " ".join(value.split())
    return text[: limit - 1] + "…" if len(text) > limit else text


def extract_langgraph_path(spans: list[dict[str, Any]]) -> list[str]:
    path = []
    ordered = sorted(spans, key=lambda span: int(span.get("start_time_unix_nano") or 0))
    for span in ordered:
        metadata = json_loads((span.get("attributes") or {}).get("metadata"), {})
        node = metadata.get("langgraph_node") if isinstance(metadata, dict) else ""
        label = str(node or span.get("name") or "")
        if label and span_type(span) in {"CHAIN", "AGENT", "TOOL"}:
            if not path or path[-1] != label:
                path.append(label)
    return path


def top_span_durations(spans: list[dict[str, Any]], n: int = 8) -> list[dict[str, Any]]:
    rows = []
    for span in spans:
        status = span.get("status") or {}
        rows.append(
            {
                "name": str(span.get("name") or ""),
                "span_type": span_type(span),
                "duration_s": ms_to_s(span_duration_ms(span)),
                "status_code": str(status.get("code") or ""),
                "status_message": short_text(status.get("message") or "", 160),
            }
        )
    return sorted(rows, key=lambda row: row["duration_s"] or 0, reverse=True)[:n]


def trace_diagnosis(row: pd.Series, raw_spans: list[dict[str, Any]]) -> str:
    errors = json_loads(row.get("span_errors_json"), [])
    retries = json_loads(row.get("lat_retries_json"), [])
    steps = json_loads(row.get("lat_steps_json"), [])

    if errors:
        first = errors[0]
        where = first.get("langgraph_node") or first.get("span_name") or "span"
        what = first.get("exception_type") or first.get("status_code") or "error"
        msg = first.get("exception_message") or first.get("status_message") or ""
        return f"Span-level error in {where}: {what}. {short_text(msg, 180)}"

    if retries:
        by_bucket = Counter(item.get("bucket") or "unknown" for item in retries)
        overhead_s = ms_to_s(row.get("lat_retry_overhead_ms")) or 0
        return f"Likely hidden LLM retry/backoff: {len(retries)} slow CHAT_MODEL call(s), ~{overhead_s}s estimated overhead; buckets {dict(by_bucket)}."

    numeric_steps = [(item.get("label"), float(item.get("ms") or 0.0)) for item in steps if isinstance(item, dict)]
    numeric_steps.sort(key=lambda item: item[1], reverse=True)
    if numeric_steps:
        label, ms = numeric_steps[0]
        return f"Longest latency bucket is {label} at {round(ms / 1000.0, 3)}s."

    top = top_span_durations(raw_spans, 1)
    if top:
        return f"Longest span is {top[0]['name']} ({top[0]['span_type']}) at {top[0]['duration_s']}s."
    return "No clear span-level cause found in the available trace payload."


raw_by_trace = {
    str(row.get("trace_id")): coerce_spans(row.get("spans"))
    for row in traces_df.to_dict(orient="records")
}

## Build detailed analysis tables

In [7]:
analysis_df = trace_level_df.copy()
analysis_df["duration_s"] = analysis_df["execution_duration_ms"].apply(ms_to_s)
analysis_df["lat_total_s"] = analysis_df["lat_total_ms"].apply(ms_to_s)
analysis_df["lat_retry_overhead_s"] = analysis_df["lat_retry_overhead_ms"].apply(ms_to_s)
analysis_df["langgraph_path"] = [
    json.dumps(extract_langgraph_path(raw_by_trace.get(str(trace_id), [])), ensure_ascii=False)
    for trace_id in analysis_df["trace_id"]
]
analysis_df["top_spans_json"] = [
    json.dumps(top_span_durations(raw_by_trace.get(str(trace_id), [])), ensure_ascii=False)
    for trace_id in analysis_df["trace_id"]
]
analysis_df["diagnosis"] = [
    trace_diagnosis(row, raw_by_trace.get(str(row["trace_id"]), []))
    for _, row in analysis_df.iterrows()
]

detail_columns = [
    "trace_id",
    "session_id",
    "request_time",
    "state",
    "duration_s",
    "lat_total_s",
    "lat_retry_overhead_s",
    "lat_retry_call_count",
    "trace_has_span_error",
    "span_error_count",
    "span_error_types_json",
    "span_errors_json",
    "diagnosis",
    "langgraph_path",
    "top_spans_json",
    "user_query",
    "actual_agent",
    "actual_agents_path",
    "actual_tool_calls",
    "actual_response",
]
detail_columns = [column for column in detail_columns if column in analysis_df.columns]
details_df = analysis_df[detail_columns].sort_values("duration_s", ascending=False)

details_path = OUT_DIR / f"latest_{N_TRACES}_trace_details.csv"
details_df.to_csv(details_path, index=False)

print(f"Wrote details: {details_path}")
details_df.head(10)

Wrote details: /private/tmp/hg_latest_trace_analysis/latest_100_trace_details.csv


,trace_id,session_id,request_time,state,duration_s,lat_total_s,lat_retry_overhead_s,lat_retry_call_count,trace_has_span_error,span_error_count,span_error_types_json,span_errors_json,diagnosis,langgraph_path,top_spans_json,user_query,actual_agent,actual_agents_path,actual_tool_calls,actual_response
18,tr-f391939b1387d15fdb0fab85d35fc3a7,086e2751-fb62-446e-94fe-4a5774247cf4,1780921053882,OK,155.394,155.395,128.373,1,False,0,[],[],Likely hidden LLM retry/backoff: 1 slow CHAT_M...,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",How much did I spend on groceries?,daily_banking_agent,[daily_banking_agent],"[{'tool': 'analyze_transactions', 'step': 1, '...","You’ve spent a total of 24,00 EUR on groceries..."
13,tr-c415c422229c4d3d22ae6a8e72a3b607,e0eb7801-d7e6-45e9-9378-1258c5bfcc31,1780922254374,OK,122.208,122.208,59.103,2,True,3,"[""CancelledError""]","[{""span_name"": ""LangGraph"", ""span_type"": ""CHAI...",Span-level error in tools: CancelledError.,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",How do I set up a standing order?,daily_banking_agent,[daily_banking_agent],"[{'tool': 'knowledge_search', 'step': 1, 'argu...",You set up a standing order in George Mobile w...
52,tr-b58362e01ea55405a926fa8c0cc94ee8,985a877b-8707-4c0d-bac0-e9504431ca2c,1780651427143,OK,119.664,119.665,104.989,3,False,0,[],[],Likely hidden LLM retry/backoff: 3 slow CHAT_M...,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",What do I spend on groceries?,daily_banking_agent,[daily_banking_agent],"[{'tool': 'analyze_transactions', 'step': 1, '...",I currently can’t show the exact amounts you s...
53,tr-2a83d8e0e1a5441cb5c9ad570d7519b4,985a877b-8707-4c0d-bac0-e9504431ca2c,1780651406279,OK,85.912,85.913,63.535,2,False,0,[],[],Likely hidden LLM retry/backoff: 2 slow CHAT_M...,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",What do I spend on groceries?,daily_banking_agent,[daily_banking_agent],"[{'tool': 'analyze_transactions', 'step': 1, '...",Momentálne tu nevidím žiadne zaúčtované výdavk...
12,tr-f0eff9f608c73b14591895552a826297,85b58259-e78d-4590-86bf-2f484c55d878,1780922773482,OK,52.909,52.910,0.000,0,False,0,[],[],Longest latency bucket is overhead at 52.91s.,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",Should I buy NVIDIA?,main_agent,[main_agent],[],I can’t give personalized investment advice or...
99,tr-3720bdcc1cfb4bb5652c269e350eb32e,ea1d1822-abb8-4827-b549-5cb1e2059ef8,1780484958147,OK,48.206,48.206,0.000,0,True,2,"[""ToolError"", ""STATUS_CODE_ERROR""]","[{""span_name"": ""analyze_transactions"", ""span_t...",Span-level error in tools: ToolError. Error ca...,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",,daily_banking_agent,[daily_banking_agent],"[{'tool': 'george-gcg-product_getCards', 'step...",I can’t teraz priamo načítať tvoje konkrétne k...
64,tr-279988e72bbcbe8371a3d12b89b77248,ae7de43b-224e-41c9-b9ce-139e460b87e3,1780584346486,OK,44.887,44.887,0.000,0,False,0,[],[],Longest latency bucket is tools at 34.7s.,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",review transactions and show me how i can save,daily_banking_agent,[daily_banking_agent],"[{'tool': 'analyze_transactions', 'step': 1, '...",Here’s a quick overview of where your money is...
87,tr-d707efa09f1fc2296f8db91dedc707b5,1bbaf8ce-f1dc-4c1e-8893-74e6b45b6b21,1780556108627,OK,42.772,42.773,0.000,0,False,0,[],[],Longest latency bucket is tools at 30.291s.,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",,daily_banking_agent,[d

## Session rollup

This helps spot whether the long sessions are one huge trace or repeated slow traces inside the same conversation/session.

In [8]:
session_rows = []
if "session_id" in analysis_df.columns:
    for session_id, group in analysis_df.groupby("session_id", dropna=False):
        session_rows.append(
            {
                "session_id": session_id,
                "trace_count": len(group),
                "total_duration_s": round(group["duration_s"].fillna(0).sum(), 3),
                "max_duration_s": round(group["duration_s"].fillna(0).max(), 3),
                "span_error_traces": int(group["trace_has_span_error"].fillna(False).sum()),
                "retry_traces": int((group["lat_retry_call_count"].fillna(0) > 0).sum()),
                "trace_ids": ", ".join(group["trace_id"].astype(str).tolist()[:10]),
            }
        )

sessions_df = pd.DataFrame(session_rows)
if not sessions_df.empty:
    sessions_df = sessions_df.sort_values(["total_duration_s", "max_duration_s"], ascending=False)

sessions_path = OUT_DIR / f"latest_{N_TRACES}_session_rollup.csv"
sessions_df.to_csv(sessions_path, index=False)

print(f"Wrote session rollup: {sessions_path}")
sessions_df.head(10)

Wrote session rollup: /private/tmp/hg_latest_trace_analysis/latest_100_session_rollup.csv


,session_id,trace_count,total_duration_s,max_duration_s,span_error_traces,retry_traces,trace_ids
31,985a877b-8707-4c0d-bac0-e9504431ca2c,2,205.576,119.664,0,2,"tr-b58362e01ea55405a926fa8c0cc94ee8, tr-2a83d8..."
2,086e2751-fb62-446e-94fe-4a5774247cf4,1,155.394,155.394,0,1,tr-f391939b1387d15fdb0fab85d35fc3a7
50,e0eb7801-d7e6-45e9-9378-1258c5bfcc31,2,130.586,122.208,1,1,"tr-c415c422229c4d3d22ae6a8e72a3b607, tr-b9b54d..."
34,9dedcd4b-e1de-4db7-867f-211e78de79ac,5,71.998,22.450,0,0,"tr-c163bc15de2d2712420b57f25cda85c4, tr-9c1ff0..."
52,e13dc026-9092-470a-9786-b36adf3e1e35,4,69.917,29.948,0,0,"tr-02c125cc2b011896bda14641f4d43bf4, tr-f74210..."
28,85b58259-e78d-4590-86bf-2f484c55d878,2,60.845,52.909,0,0,"tr-35ae21a0c3b6f81c5b3c8781c4e4fcb6, tr-f0eff9..."
24,712ba4bc-0cc3-40ca-bffd-877d5cb978cd,2,57.316,39.291,0,0,"tr-cd7f4f8e459ba2148abf01221f5cf804, tr-a65b66..."
47,d9376f74-b474-4035-9614-347fbaa9b616,5,54.557,16.152,0,0,"tr-a896441527bcef26bdcc1b623223bee7, tr-4c47fe..."
7,25ed0e6e-584e-40a0-af9b-f722ca23c24d,5,51.045,17.730,0,0,"tr-dab30dd2b1ff8342c46337866b15b5c6, tr-19cbca..."
57,ea1d1822-abb8-4827-b549-5cb1e2059ef8,1,48.206,48.206,1,0,tr-3720bdcc1cfb4bb5652c269e350eb32e


## Summary findings

In [9]:
state_counts = analysis_df["state"].fillna("").astype(str).value_counts().to_dict() if "state" in analysis_df else {}
span_error_count = int(analysis_df["trace_has_span_error"].fillna(False).sum()) if "trace_has_span_error" in analysis_df else 0
retry_trace_count = int((analysis_df["lat_retry_call_count"].fillna(0) > 0).sum()) if "lat_retry_call_count" in analysis_df else 0

span_error_type_counts = Counter(
    error_type
    for raw in analysis_df.get("span_error_types_json", pd.Series(dtype=str)).tolist()
    for error_type in json_loads(raw, [])
)

latency_summary = {
    "min_s": ms_to_s(analysis_df["lat_total_ms"].min()),
    "p50_s": ms_to_s(analysis_df["lat_total_ms"].quantile(0.50)),
    "p90_s": ms_to_s(analysis_df["lat_total_ms"].quantile(0.90)),
    "p95_s": ms_to_s(analysis_df["lat_total_ms"].quantile(0.95)),
    "max_s": ms_to_s(analysis_df["lat_total_ms"].max()),
}

summary = {
    "experiment_id": EXPERIMENT_ID,
    "traces_fetched": int(len(traces_df)),
    "parsed_rows": int(len(parsed_df)),
    "state_counts": state_counts,
    "span_error_trace_count": span_error_count,
    "span_error_type_counts": dict(span_error_type_counts),
    "retry_trace_count": retry_trace_count,
    "latency_seconds": latency_summary,
    "details_csv": str(details_path),
    "sessions_csv": str(sessions_path),
    "raw_pickle": str(raw_pickle),
}

summary_path = OUT_DIR / "summary.json"
with summary_path.open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, ensure_ascii=False, indent=2)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print(f"Summary written to: {summary_path}")

{
  "experiment_id": "2373724612324597",
  "traces_fetched": 100,
  "parsed_rows": 100,
  "state_counts": {
    "OK": 99,
    "ERROR": 1
  },
  "span_error_trace_count": 4,
  "span_error_type_counts": {
    "CancelledError": 2,
    "ValidationError": 1,
    "ToolError": 1,
    "STATUS_CODE_ERROR": 1
  },
  "retry_trace_count": 6,
  "latency_seconds": {
    "min_s": 0.283,
    "p50_s": 16.056,
    "p90_s": 39.094,
    "p95_s": 48.441,
    "max_s": 155.395
  },
  "details_csv": "/private/tmp/hg_latest_trace_analysis/latest_100_trace_details.csv",
  "sessions_csv": "/private/tmp/hg_latest_trace_analysis/latest_100_session_rollup.csv",
  "raw_pickle": "/private/tmp/hg_latest_trace_analysis/raw_latest_100_traces.pkl"
}
Summary written to: /private/tmp/hg_latest_trace_analysis/summary.json


## Detailed trace lists

In [10]:
# Traces with either MLflow state != OK or at least one span-level error.
error_details_df = details_df[
    (details_df.get("trace_has_span_error", False) == True)
    | (details_df.get("state", "").astype(str).str.upper() != "OK")
]

print(f"Error traces: {len(error_details_df):,}")
error_details_df

Error traces: 4


,trace_id,session_id,request_time,state,duration_s,lat_total_s,lat_retry_overhead_s,lat_retry_call_count,trace_has_span_error,span_error_count,span_error_types_json,span_errors_json,diagnosis,langgraph_path,top_spans_json,user_query,actual_agent,actual_agents_path,actual_tool_calls,actual_response
13,tr-c415c422229c4d3d22ae6a8e72a3b607,e0eb7801-d7e6-45e9-9378-1258c5bfcc31,1780922254374,OK,122.208,122.208,59.103,2,True,3,"[""CancelledError""]","[{""span_name"": ""LangGraph"", ""span_type"": ""CHAI...",Span-level error in tools: CancelledError.,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",How do I set up a standing order?,daily_banking_agent,[daily_banking_agent],"[{'tool': 'knowledge_search', 'step': 1, 'argu...",You set up a standing order in George Mobile w...
99,tr-3720bdcc1cfb4bb5652c269e350eb32e,ea1d1822-abb8-4827-b549-5cb1e2059ef8,1780484958147,OK,48.206,48.206,0.000,0,True,2,"[""ToolError"", ""STATUS_CODE_ERROR""]","[{""span_name"": ""analyze_transactions"", ""span_t...",Span-level error in tools: ToolError. Error ca...,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",,daily_banking_agent,[daily_banking_agent],"[{'tool': 'george-gcg-product_getCards', 'step...",I can’t teraz priamo načítať tvoje konkrétne k...
17,tr-25929c8b450e1b0d48dfa0e146d54043,695abd67-0a47-492e-8f61-c84963f533ba,1780921747078,ERROR,26.122,26.123,23.233,1,True,1,"[""CancelledError""]","[{""span_name"": ""LangGraph"", ""span_type"": ""CHAI...",Span-level error in LangGraph: CancelledError....,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",When does my card expire?,daily_banking_agent,[daily_banking_agent],[],
65,tr-00541cab692922ff6e6b7ee2c652fbc2,e5ce588e-71bb-4edc-bf0f-a14cbc4143a0,1780584029649,OK,24.492,24.493,0.000,0,True,1,"[""ValidationError""]","[{""span_name"": ""analyze_transactions"", ""span_t...",Span-level error in tools: ValidationError. 1 ...,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",how much did i spend in croatia,daily_banking_agent,[daily_banking_agent],"[{'tool': 'analyze_transactions', 'step': 1, '...",I can’t calculate the exact amount for Croatia...


In [11]:
# Longest traces, including trace id and plain-English diagnosis.
longest_columns = [
    "trace_id",
    "session_id",
    "duration_s",
    "lat_retry_overhead_s",
    "lat_retry_call_count",
    "trace_has_span_error",
    "span_error_count",
    "diagnosis",
    "langgraph_path",
    "top_spans_json",
    "user_query",
]
longest_columns = [column for column in longest_columns if column in details_df.columns]
details_df[longest_columns].head(20)

,trace_id,session_id,duration_s,lat_retry_overhead_s,lat_retry_call_count,trace_has_span_error,span_error_count,diagnosis,langgraph_path,top_spans_json,user_query
18,tr-f391939b1387d15fdb0fab85d35fc3a7,086e2751-fb62-446e-94fe-4a5774247cf4,155.394,128.373,1,False,0,Likely hidden LLM retry/backoff: 1 slow CHAT_M...,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",How much did I spend on groceries?
13,tr-c415c422229c4d3d22ae6a8e72a3b607,e0eb7801-d7e6-45e9-9378-1258c5bfcc31,122.208,59.103,2,True,3,Span-level error in tools: CancelledError.,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",How do I set up a standing order?
52,tr-b58362e01ea55405a926fa8c0cc94ee8,985a877b-8707-4c0d-bac0-e9504431ca2c,119.664,104.989,3,False,0,Likely hidden LLM retry/backoff: 3 slow CHAT_M...,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",What do I spend on groceries?
53,tr-2a83d8e0e1a5441cb5c9ad570d7519b4,985a877b-8707-4c0d-bac0-e9504431ca2c,85.912,63.535,2,False,0,Likely hidden LLM retry/backoff: 2 slow CHAT_M...,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",What do I spend on groceries?
12,tr-f0eff9f608c73b14591895552a826297,85b58259-e78d-4590-86bf-2f484c55d878,52.909,0.000,0,False,0,Longest latency bucket is overhead at 52.91s.,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",Should I buy NVIDIA?
99,tr-3720bdcc1cfb4bb5652c269e350eb32e,ea1d1822-abb8-4827-b549-5cb1e2059ef8,48.206,0.000,0,True,2,Span-level error in tools: ToolError. Error ca...,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",
64,tr-279988e72bbcbe8371a3d12b89b77248,ae7de43b-224e-41c9-b9ce-139e460b87e3,44.887,0.000,0,False,0,Longest latency bucket is tools at 34.7s.,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",review transactions and show me how i can save
87,tr-d707efa09f1fc2296f8db91dedc707b5,1bbaf8ce-f1dc-4c1e-8893-74e6b45b6b21,42.772,0.000,0,False,0,Longest latency bucket is tools at 30.291s.,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",
88,tr-4693ba8d7fd0545d226f02b190143c88,1bbaf8ce-f1dc-4c1e-8893-74e6b45b6b26,39.313,0.000,0,False,0,Longest latency bucket is tools at 32.472s.,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...",
97,tr-cd7f4f8e459ba2148abf01221f5cf804,712ba4bc-0cc3-40ca-bffd-877d5cb978cd,39.291,0.000,0,False,0,Longest latency bucket is tools at 29.906s.,"[""LangGraph"", ""prompt_shield"", ""orchestration""...","[{""name"": ""LangGraph"", ""span_type"": ""CHAIN"", ""...","what are m cards, my accounts, and how can i i..."
